# Credit Scoring EDA

Notebook ini merangkum EDA singkat untuk `data_C.csv` dan alasan perubahan pipeline training.

Tujuan utama:
- menemukan cleaning yang paling berdampak,
- mengurangi fitur yang membuat training berat,
- menaikkan metrik model tanpa membuat training terlalu lama.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from credit_scoring.features import build_features

raw = pd.read_csv(PROJECT_ROOT / "data_C.csv", low_memory=False)
df = build_features(raw)
raw.shape, df.shape

## Ringkasan Dataset

Hasil EDA di local environment :

- Jumlah data: 25.000 rows, 29 kolom raw.
- Target tidak seimbang: `Standard` 53,13%, `Poor` 29,07%, `Good` 17,80%.
- Karena target imbalance, `macro F1` tetap dipakai untuk memilih model terbaik. Accuracy tetap dilaporkan karena mudah dibaca.

In [ ]:
target_dist = df["Credit_Score"].value_counts(normalize=True).rename("ratio")
target_count = df["Credit_Score"].value_counts().rename("count")
pd.concat([target_count, target_dist], axis=1)

## Temuan Cleaning

Beberapa nilai raw berisi noise yang perlu dibersihkan:

- `Occupation` punya placeholder `_______` sebanyak 1.765 rows.
- `Payment_Behaviour` punya placeholder `!@9#%8` sebanyak 1.951 rows.
- Ada outlier numerik yang tidak realistis, misalnya `Age=8697`, `Interest_Rate=5797`, `Num_of_Loan=-100/1495`, dan `Annual_Income=24160009`.
- `Type_of_Loan` punya 5.731 kategori unik, terlalu mahal kalau di-one-hot langsung.

In [ ]:
feature_cols = [col for col in df.columns if col != "Credit_Score"]
missing_rate = df[feature_cols].isna().mean().sort_values(ascending=False)
missing_rate.head(15).to_frame("missing_rate")

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols].quantile([0, 0.01, 0.5, 0.99, 1]).T.round(2)

In [ ]:
categorical_cols = ["Occupation", "Type_of_Loan", "Credit_Mix", "Payment_of_Min_Amount", "Payment_Behaviour", "Month"]
df[categorical_cols].nunique(dropna=True).sort_values(ascending=False).to_frame("unique_values")

## Perubahan Pipeline Dari EDA

Perubahan yang diterapkan:

- Placeholder underscore-only dan `!@9#%8` dijadikan missing value.
- Outlier domain numerik dijadikan missing value agar `SimpleImputer` menangani nilainya.
- Raw `Type_of_Loan` dikeluarkan dari categorical one-hot karena cardinality sangat tinggi.
- `Type_of_Loan` diganti menjadi fitur yang lebih kecil: jumlah tipe pinjaman dan flag per jenis pinjaman utama.
- Ditambahkan rasio finansial: `Debt_To_Income`, `EMI_To_Salary`, `Investment_To_Salary`, dan `Balance_To_Salary`.
- Kandidat model diringkas ke `extra_trees` dan `hist_gradient_boosting`.
- Default cross-validation dipercepat dari 5 fold ke 3 fold.

## Hasil Benchmark

Baseline sebelumnya:

| Model | Accuracy | Macro F1 |
|---|---:|---:|
| gradient_boosting | 0.6958 | 0.6719 |

Pipeline baru, training final sekitar 54 detik di env `AIGacor`:

| Model | CV Macro F1 | Accuracy | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|
| extra_trees | 0.6821 | 0.7138 | 0.7017 | 0.7174 |
| hist_gradient_boosting | 0.6785 | 0.7158 | 0.6951 | 0.7158 |

Model final tetap `extra_trees` karena macro F1 lebih tinggi, walaupun `hist_gradient_boosting` punya accuracy sedikit lebih tinggi.

In [ ]:
pd.read_csv(PROJECT_ROOT / "artifacts" / "experiment_results.csv")